[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ScriptsRemote/Transcend_SummerSchool/blob/main/Day4_LULC_and_Climate_with_RS/04_climate_products_et_and_water_balance.ipynb)

# Day 4 - Land Use / Land Cover & Climate with RS
## Session 4: Climate Products, ET and Water Balance

| | |
|---|---|
| **Fecha / Date** | 2026-10-15 (Thursday / Jueves) |
| **Horario / Time** | 15:40 - 17:10 |
| **Entidad / Entity** | UoM |
| **Instructores / Instructors** | Christhian |

### Contenido / Content

> Earth Observation climate products (CHIRPS, GSMaP, IMERG, ERA5-Land). The MODIS MOD16 ET
> product (structure, scaling, resolution). Monthly time series, simplified water balance
> (P - ET), zonal statistics and trend analysis for the Lake Titicaca watershed.

### Objetivos de aprendizaje / Learning objectives

- [ ] Write the water balance equation and explain what the simplified **P - ET** represents
- [ ] Compare the main satellite **precipitation** products by spatial and temporal resolution,
      record length and method
- [ ] Explain how **evapotranspiration** is estimated from space and know the main ET datasets,
      inside and outside Google Earth Engine
- [ ] Build monthly P, ET and P - ET collections in GEE (CHIRPS + MOD16A2GF), extract the
      watershed mean as a `DataFrame` and plot monthly and annual series with a trend line

### Datos necesarios / Required data

- `UCSB-CHG/CHIRPS/DAILY` and `MODIS/061/MOD16A2GF` from the Earth Engine catalogue
- The watershed boundary `projects/ee-christhianscunha/assets/transcend/TDP_map`
  (Lake Titicaca - Desaguadero - Poopó system)

### Lo que ya vimos / What is assumed

Days 2 and 3 covered image collections, filters, reducers and zonal statistics. This session
applies them to **climate time series** instead of a single image.

---

## 4.1 - Teoria / Theory: the water balance

The water balance is the **accounting of the water** that enters and leaves an area - a field,
a basin, a lake - over a period of time:

$$P = ET + Q + \Delta S$$

| Term | Meaning | Typical source |
|---|---|---|
| $P$ | **Precipitation** - rain, snow, hail | rain gauges, satellites, reanalysis |
| $ET$ | **Evapotranspiration** - water returned to the atmosphere | models driven by satellite data |
| $Q$ | **Runoff** - water leaving the area by rivers (and groundwater) | stream gauges |
| $\Delta S$ | **Change in storage** - soil moisture, groundwater, lakes, snow | GRACE, lake levels, soil moisture |

### The simplified balance: P - ET

$Q$ and $\Delta S$ are hard to measure everywhere. P and ET, on the other hand, are available as
global gridded products. Their difference

$$P - ET = Q + \Delta S$$

is the water **left over** after the atmosphere takes its share:

- **P - ET > 0** - surplus: water available to recharge soils and aquifers and to feed rivers
  and the lake;
- **P - ET < 0** - deficit: the surface loses more than it receives, and the difference comes
  from **storage** (soil, groundwater, the lake itself).

Over a full year $\Delta S$ is often close to zero, so annual P - ET is an estimate of the
**water yield** of the basin.

### Units

Everything is expressed as a **depth of water in mm** per period (mm/month, mm/year).
1 mm over 1 m² is 1 litre; **1 mm over 1 km² = 1,000 m³**. This makes P, ET and P - ET
directly comparable and easy to convert to volumes.

### The Titicaca context

The altiplano has a strongly **seasonal** climate: most of the rain falls between December and
March, and May to September is almost dry. Monthly P - ET is therefore positive in the wet
season and negative in the dry season. Lake Titicaca itself loses most of its water by
**evaporation** - only a small part leaves through the Desaguadero river - so small changes in
P or ET have a direct effect on the lake level.

---

## 4.2 - Teoria / Theory: precipitation from space

### How precipitation is estimated

| Source | How it works | Strength | Limitation |
|---|---|---|---|
| **Rain gauges** | direct measurement at a point | the reference ("truth") | few stations, especially in the Andes |
| **Infrared (IR)** - geostationary satellites | cold cloud tops = probable rain | every 10-30 min, long records | indirect: cold clouds do not always rain |
| **Passive microwave (PMW)** - polar satellites | signal of rain drops and ice inside the cloud | physically closer to rain | few overpasses per day |
| **Radar** - GPM Core Observatory | 3D structure of the rain | most accurate from space | narrow swath, calibration only |
| **Reanalysis** - ERA5 | weather model + all observations | complete in space and time, many variables | coarse; it is a model, not an observation |

The global products **combine** these sources: IR gives the frequency, PMW gives the rain
intensity, and gauges correct the bias.

### Main products in Google Earth Engine

| Product | GEE ID | Pixel | Time step | Record | Method |
|---|---|---|---|---|---|
| **CHIRPS v2** | `UCSB-CHG/CHIRPS/DAILY` | 0.05° (~5.5 km) | daily | 1981 - present | IR + climatology + **gauges**, 50°S-50°N |
| **CHIRPS v3** | `UCSB-CHC/CHIRPS/V3/DAILY_RNL` | 0.05° (~5.5 km) | daily | 1981 - present | new version, 60°S-60°N, daily split with ERA5 |
| **GSMaP v8** | `JAXA/GPM_L3/GSMaP/v8/operational` | 0.1° (~11 km) | **hourly** | 1998 - present | PMW + IR (JAXA) |
| **IMERG V07** | `NASA/GPM_L3/IMERG_V07` | 0.1° (~11 km) | **30 min** | 1998 - present | PMW + IR + gauges (NASA) |
| **ERA5-Land** | `ECMWF/ERA5_LAND/MONTHLY_AGGR` | 0.1° (~11 km) | hourly / monthly | 1950 - present | reanalysis (model) |
| **TerraClimate** | `IDAHO_EPSCOR/TERRACLIMATE` | ~4 km | monthly | 1958 - present | climatology + reanalysis |
| **PERSIANN-CDR** | `NOAA/PERSIANN-CDR` | 0.25° (~27 km) | daily | 1983 - present | IR + neural network, gauge-adjusted |

### How to choose

- **Spatial detail** - CHIRPS and TerraClimate (~5 km) show more detail than the 0.1° products;
  this matters in mountains, where rain changes over short distances.
- **Temporal detail** - for storms and floods you need hours (GSMaP, IMERG); for monthly
  balances, daily or monthly products are enough.
- **Record length** - for trends and climate normals (30 years) use CHIRPS, ERA5-Land or
  TerraClimate.
- **Units** - always check them: CHIRPS is mm/day, GSMaP and IMERG are **mm/hour**, ERA5-Land
  is in **metres**. A monthly total is the **sum** of the daily (or hourly) values.

> **In the Andes**, all products have larger errors: complex terrain, convective storms, hail
> and few gauges to correct them. Whenever possible, **compare with local stations**
> (SENAMHI Bolivia / Peru) before trusting a single product.

In the practice we use **CHIRPS**: daily, ~5.5 km and corrected with gauges.

---

## 4.3 - Teoria / Theory: evapotranspiration (ET)

**Evapotranspiration** is the sum of two processes that return water to the atmosphere:

- **Evaporation (E)** - from soil, open water and wet surfaces;
- **Transpiration (T)** - water taken up by roots and released through the leaves.

Two concepts must not be confused:

- **Potential ET (PET)** - how much **could** evaporate if water were unlimited; depends only on
  the atmosphere (radiation, temperature, wind, humidity).
- **Actual ET (AET or ETa)** - how much **really** evaporates; limited by the water available
  in the soil and by the vegetation. **The water balance uses actual ET.**

On the altiplano PET is high (strong radiation, dry air), but in the dry season actual ET is
low because there is little water left in the soil.

### How ET is estimated from space

ET cannot be seen directly by a satellite. It is **modelled** from variables that can be
observed:

| Approach | Idea | Examples |
|---|---|---|
| **Penman-Monteith** | energy + vegetation (LAI, albedo) + weather data | MOD16, PML_V2 |
| **Surface energy balance** (thermal band) | the energy not used to heat the surface is used to evaporate water: a cooler surface evaporates more | SEBAL, METRIC, **SSEBop** (Landsat) |
| **Land surface models / reanalysis** | a hydrological model run with weather data | GLDAS, FLDAS, ERA5-Land, TerraClimate |

### MOD16 - the product we use

`MODIS/061/MOD16A2GF` is the **gap-filled** version of MOD16A2 (Terra satellite):

| Property | Value |
|---|---|
| Pixel | **500 m** |
| Time step | **8-day composite** - each image is the **total** ET of 8 days (the last one of the year has 5 or 6) |
| Band | `ET`, units **kg/m²/8 days** = mm/8 days |
| Scale factor | **0.1** - multiply before using |
| Gap filling | done once per year, at year end - so the most recent months may be missing |
| Masked areas | **no values over water, urban and barren surfaces** - including Lake Titicaca |

The last point is important for the practice: MOD16 has **no ET for the lake**, so we compute
the balance only over the pixels where MOD16 has data (the land part of the watershed).

### Main ET datasets in Google Earth Engine

| Dataset | GEE ID | Pixel | Time step | Coverage | Method |
|---|---|---|---|---|---|
| **MOD16A2GF v061** | `MODIS/061/MOD16A2GF` | 500 m | 8-day | global, 2000 - last full year | Penman-Monteith, gap-filled |
| **MOD16A2 v061** | `MODIS/061/MOD16A2` | 500 m | 8-day | global, near real time | Penman-Monteith (with gaps) |
| **PML_V2.2a** | `projects/pml_evapotranspiration/PML/OUTPUT/PML_V22a` | 500 m | 8-day | global | Penman-Monteith-Leuning, coupled with GPP |
| **TerraClimate** | `IDAHO_EPSCOR/TERRACLIMATE` (band `aet`) | ~4 km | monthly | global, 1958 - | water balance model |
| **ERA5-Land** | `ECMWF/ERA5_LAND/MONTHLY_AGGR` | ~11 km | monthly | global, 1950 - | reanalysis (evaporation is **negative**, in metres) |
| **FLDAS** | `NASA/FLDAS/NOAH01/C/GL/M/V001` | ~11 km | monthly | global, 1982 - | land surface model |
| **GLDAS-2.1** | `NASA/GLDAS/V021/NOAH/G025/T3H` | ~28 km | 3-hourly | global, 2000 - | land surface model |
| **OpenET** | `projects/openet/...` | **30 m** | monthly | **USA only** | ensemble of Landsat models |

### Landsat ET - outside Google Earth Engine

At 500 m, a MOD16 pixel mixes fields, roads and houses. For **field-scale** work (irrigation,
crops) ET is needed at the Landsat resolution:

- **Landsat Collection 2 Level-3 Provisional Actual Evapotranspiration** (USGS) - daily actual
  ET in mm for the date of each Landsat scene, at **30 m**, computed with the **SSEBop** surface
  energy balance model from the thermal band. Coverage is **global**, but the product is **not
  in the GEE catalogue**: it is ordered scene by scene through the USGS ESPA on-demand
  interface. In the USA, the equivalent Landsat-based monthly ET is available in GEE through
  OpenET.
- Other well-known products outside GEE: **GLEAM** (global, ~25 km / ~10 km, daily) and FAO
  **WaPOR** (Africa and the Near East, 250 m down to 30 m).

> **Trade-off.** Landsat ET has fine detail but only one value every 8-16 days (and none under
> clouds). MOD16 has an image every 8 days but at 500 m. For a monthly water balance of a
> large basin like Titicaca, MOD16 is the practical choice.

---

## 4.4 - Practica guiada / Guided hands-on: water balance of the Titicaca watershed

Workflow (adapted from the *Cloud-Based Remote Sensing with Google Earth Engine* book):

1. watershed and period;
2. CHIRPS (P) and MOD16 (ET), with the scale factor;
3. **monthly** P, ET and P - ET as one image collection;
4. watershed mean of each month -> `DataFrame`;
5. charts: monthly balance, monthly P and ET, annual totals with trend;
6. maps of the mean annual P, ET and P - ET.

In [ ]:
# Libraries (uncomment the pip line on a fresh runtime, e.g. Google Colab)
#%pip install -q earthengine-api geemap

import ee                          # Google Earth Engine API
import geemap                      # interactive maps for Earth Engine
import numpy as np                 # trend line
import pandas as pd                # tables (DataFrames)
import matplotlib.pyplot as plt    # charts

In [ ]:
# Authenticate and initialise Earth Engine
# >>> REPLACE with your own Cloud project id <<<
EE_PROJECT = "ee-christhianscunha"

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

### Watershed and period

In [ ]:
# Lake Titicaca watershed (TDP system)
roi = ee.FeatureCollection("projects/ee-christhianscunha/assets/transcend/TDP_map")
basin = roi.geometry()

# Period: complete years
START_YEAR, END_YEAR = 2020, 2025
start = ee.Date.fromYMD(START_YEAR, 1, 1)
end = ee.Date.fromYMD(END_YEAR + 1, 1, 1)      # 1 January of the next year

Map = geemap.Map()
Map.centerObject(roi, 7)
Map.addLayer(roi.style(color="red", fillColor="00000000", width=2), {}, "Lake Titicaca watershed")
Map

### Precipitation (CHIRPS) and ET (MOD16)

- CHIRPS: band `precipitation`, **mm/day** - no scale factor.
- MOD16A2GF: band `ET`, **kg/m²/8 days** with scale factor **0.1** -> mm/8 days.

In [ ]:
# Precipitation: CHIRPS daily (mm/day)
chirps = (ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
            .select("precipitation")
            .filterDate(start, end))


def scale_mod16(image):
    # Scale factor 0.1 -> mm per 8 days; keep the date of the image
    return image.multiply(0.1).copyProperties(image, ["system:time_start"])


# Evapotranspiration: MOD16A2GF 8-day composites (mm/8 days)
mod16 = (ee.ImageCollection("MODIS/061/MOD16A2GF")
           .select("ET")
           .filterDate(start, end)
           .map(scale_mod16))

print("CHIRPS images:", chirps.size().getInfo())
print("MOD16 images :", mod16.size().getInfo())
print("Last MOD16 date:", ee.Date(mod16.aggregate_max("system:time_start")).format("YYYY-MM-dd").getInfo())

### Monthly P, ET and P - ET

For each month we **sum** all daily CHIRPS images and all MOD16 composites that start in that
month, and subtract them. One function builds the three bands of one month; `map()` applies it
to every month of the period.

Two details:

- P is masked with the ET mask (`updateMask`), so both are computed over the **same pixels**
  (MOD16 has no data over the lake and urban areas).
- Summing the 8-day composites that **start** in a month is an approximation: a month gets 3 or
  4 composites, and one composite can cross into the next month. For monthly and annual
  balances the error is small, but it shows up as month-to-month noise.

In [ ]:
def monthly_balance(i):
    # First and last day of month number i of the period
    month_start = start.advance(i, "month")
    month_end = month_start.advance(1, "month")

    # Monthly totals (mm/month)
    et = mod16.filterDate(month_start, month_end).sum().rename("ET")
    p = chirps.filterDate(month_start, month_end).sum().rename("precipitation")
    p = p.updateMask(et.mask())                           # same pixels as ET

    # Simplified water balance
    wb = p.subtract(et).rename("water_balance")

    return (p.addBands([et, wb])
             .set("system:time_start", month_start.millis())
             .set("year", month_start.get("year"))
             .set("month", month_start.get("month")))


# One image per month: 6 years x 12 months = 72 images
n_months = (END_YEAR - START_YEAR + 1) * 12
monthly = ee.ImageCollection(ee.List.sequence(0, n_months - 1).map(monthly_balance))

print("Monthly images:", monthly.size().getInfo())
print("Bands         :", monthly.first().bandNames().getInfo())

### Zonal statistics: watershed mean of each month

`reduceRegion` with the **mean** reducer gives one value per band and month for the whole
watershed. We store each result as a `Feature` and convert the collection into a `DataFrame`
with geemap.

In [ ]:
def watershed_mean(image):
    # Mean of P, ET and P - ET over the watershed (500 m = MOD16 pixel)
    stats = image.reduceRegion(reducer=ee.Reducer.mean(), geometry=basin,
                               scale=500, maxPixels=1e10, tileScale=4)
    return ee.Feature(None, stats).set("date", image.date().format("YYYY-MM-dd"))


table = ee.FeatureCollection(monthly.map(watershed_mean))

# To a DataFrame, sorted by date
df = geemap.ee_to_df(table)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").set_index("date")[["precipitation", "ET", "water_balance"]]

df.round(1).head(12)

### Chart 1 - Monthly water balance

Bars: P - ET (positive = surplus, negative = deficit). Lines: P and ET.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))

# P - ET as bars, P and ET as lines
ax.bar(df.index, df["water_balance"], width=20, color="#eb9800", label="Water balance (P - ET)")
ax.plot(df.index, df["precipitation"], color="#25a6d7", marker="o", markersize=4, label="Precipitation (CHIRPS)")
ax.plot(df.index, df["ET"], color="#757373", marker="v", markersize=4, label="ET (MOD16)")
ax.axhline(0, color="black", linewidth=0.8)

ax.set_title("Monthly water balance - Lake Titicaca watershed")
ax.set_xlabel("Date")
ax.set_ylabel("mm/month")
ax.legend(frameon=False, ncol=3, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

**How to read it.** Every year repeats the same pattern: a **surplus** in the rainy season
(December-March), when P is far above ET, and a **deficit** from May to October, when ET
continues but rain stops. Compare the height of the rainy-season bars from year to year - a
weak rainy season means less water to recharge the system and the lake.

### Chart 2 - Monthly precipitation and ET

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Monthly precipitation
ax1.bar(df.index, df["precipitation"], width=20, color="#25a6d7")
ax1.set_title(f"Monthly precipitation (CHIRPS) - {START_YEAR}-{END_YEAR}")
ax1.set_ylabel("P (mm/month)")

# Monthly evapotranspiration
ax2.bar(df.index, df["ET"], width=20, color="#757373")
ax2.set_title(f"Monthly evapotranspiration (MOD16) - {START_YEAR}-{END_YEAR}")
ax2.set_ylabel("ET (mm/month)")
ax2.set_xlabel("Date")

for ax in (ax1, ax2):
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

Notice that **ET follows P with a delay**: it rises with the first rains and stays relatively
high for some months after the rain stops, while the soil and the vegetation still have water.

### Chart 3 - Annual totals and trend

Annual totals are the **sum** of the 12 months. A straight line fitted with `np.polyfit` shows
the direction of change (mm per year).

In [ ]:
# Annual totals (mm/year)
annual = df.groupby(df.index.year).sum()
annual.index.name = "year"
annual.round(0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, band, colour, name in [(axes[0], "precipitation", "#25a6d7", "Precipitation"),
                               (axes[1], "ET", "#757373", "Evapotranspiration")]:
    # Bars: annual total
    ax.bar(annual.index, annual[band], color=colour, width=0.6)

    # Linear trend: slope in mm/year
    slope, intercept = np.polyfit(annual.index, annual[band], 1)
    ax.plot(annual.index, slope * annual.index + intercept, color="#d62728", linewidth=2.5,
            label=f"trend: {slope:+.1f} mm/year")

    ax.set_title(f"Annual {name.lower()} - Lake Titicaca watershed")
    ax.set_xlabel("Year")
    ax.set_ylabel(f"{name} (mm/year)")
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

> **Careful with trends.** Six years are enough to see the year-to-year variability (El Niño /
> La Niña years are clearly different) but **not** to conclude about climate change. Trends
> need long records (20-30 years) and a statistical test, such as Mann-Kendall.

### Maps - mean annual P, ET and P - ET

The charts summarise the watershed in one number per month. The maps show **where** the water
surplus and deficit are.

In [ ]:
# Mean annual total = sum of all months / number of years
n_years = END_YEAR - START_YEAR + 1
annual_mean = monthly.sum().divide(n_years).clip(basin)

vis_p = {"bands": ["precipitation"], "min": 300, "max": 1000,
         "palette": ["#f7fbff", "#9ecae1", "#3182bd", "#08306b"]}
vis_et = {"bands": ["ET"], "min": 100, "max": 700,
          "palette": ["#ffffcc", "#a1dab4", "#41b6c4", "#225ea8"]}
vis_wb = {"bands": ["water_balance"], "min": -300, "max": 300,
          "palette": ["#b2182b", "#f4a582", "#f7f7f7", "#92c5de", "#2166ac"]}

Map = geemap.Map()
Map.centerObject(roi, 7)
Map.addLayer(annual_mean, vis_p, "Mean annual P (mm/year)", False)
Map.addLayer(annual_mean, vis_et, "Mean annual ET (mm/year)", False)
Map.addLayer(annual_mean, vis_wb, "Mean annual P - ET (mm/year)")
Map.addLayer(roi.style(color="black", fillColor="00000000", width=1), {}, "Watershed")
Map.add_colorbar(vis_wb, label="P - ET (mm/year)")
Map

Switch the layers on and off. Where is the surplus (blue)? Usually near the **cordilleras**
and the north of the lake, where it rains more. Where is the deficit (red)? Usually on the
southern altiplano, towards Lake Poopó, where it rains little and ET is relatively high. The
lake itself is blank: MOD16 has no ET over open water.

---

---

### Water balance by land cover: agriculture and natural vegetation (MapBiomas Bolivia)

So far we averaged the whole watershed. Now we ask: **how much P, ET and P - ET is there
where people farm, and how much in the natural vegetation?** We use the land use and land cover
map of **MapBiomas Bolivia - Collection 3 (1985-2024)**, the same asset used by the
[MapBiomas user toolkit](https://github.com/mapbiomas/user-toolkit), and turn two groups of
classes into **masks** for the monthly collection we already built.

- MapBiomas stores one band per year, named `classification_YYYY`, at **30 m**.
- On the altiplano, most farming is small plots mixed with grazing, which MapBiomas maps as
  **21 - Mosaic of Uses**, not as 18 - Agriculture. That is why we first **look at which classes
  exist** in the watershed before choosing them.
- MapBiomas **Bolivia** covers only the Bolivian side of the watershed (the Peruvian side has its
  own map, MapBiomas Peru).

In [ ]:
# MapBiomas Bolivia - Collection 3: one band per year (classification_1985 ... classification_2024)
mapbiomas = ee.Image("projects/mapbiomas-public/assets/bolivia/lulc/collection3/mapbiomas_bolivia_collection3_integration_v1")

LULC_YEAR = 2024                     # last year of Collection 3
lulc = mapbiomas.select(f"classification_{LULC_YEAR}").clip(basin)

# MapBiomas Bolivia legend (official legend of Collection 4: Leyenda_Code_Bolivia_Col4.pdf)
LEGEND = {
    # 1. Forest formation
    3: "Forest", 4: "Open forest", 6: "Flooded forest",
    # 2. Grassland and shrubland
    11: "Flooded grassland/shrubland", 12: "Grassland/shrubland", 29: "Rocky outcrop",
    66: "Scrublands", 77: "Open shrublands (beta)", 63: "Steppe (beta)",
    81: "Andean grassland and shrubland", 82: "Flooded Andean grassland and shrubland",
    13: "Other non-forest natural formation",
    # 3. Farming
    15: "Pasture", 18: "Agriculture", 39: "Soybean (beta)", 40: "Flooded rice (beta)",
    72: "Other crops", 21: "Mosaic of uses",
    # 4. Non-vegetated area
    23: "Beach, dune and sand spot", 24: "Urban infrastructure", 30: "Mining", 61: "Salt flat",
    92: "Rocky surface", 68: "Other non-vegetated natural area", 25: "Other non-vegetated anthropic area",
    # 5. Water body
    33: "River, lake", 31: "Aquaculture", 34: "Glacier",
    # 6. Not observed
    27: "Not observed",
}

Read the table and choose the classes of each group. The lists below follow the MapBiomas
legend; **remove or add codes** according to what the table shows for your area.

| Group | Classes |
|---|---|
| **Agriculture** (3.2 + 3.3) | 18 Agriculture, 39 Soybean, 40 Flooded rice, 72 Other crops, 21 **Mosaic of uses** |
| **Natural vegetation** (1 + 2) | 3 Forest, 4 Open forest, 6 Flooded forest, 11 Flooded grassland/shrubland, 12 Grassland/shrubland, 66 Scrublands, 77 Open shrublands, 63 Steppe, 81 **Andean grassland and shrubland**, 82 Flooded Andean grassland and shrubland, 13 Other non-forest natural formation |

Left out on purpose: **15 Pasture** (farming, but not crops - add it to agriculture if you want all farming) and
**29 Rocky outcrop** (part of group 2 in the legend, but it is rock, not vegetation).

*Source: MapBiomas Bolivia - Códigos de las clases de cobertura y uso del suelo, Colección 4. The
codes are the same in Collection 3; classes marked (beta) may not exist in Collection 3.*

In [ ]:
# Codes from the MapBiomas Bolivia legend (see the table above)
AGRI_CLASSES = [18, 39, 40, 72, 21]                      # 3.2 Agriculture + 3.3 Mosaic of uses
VEG_CLASSES = [3, 4, 6, 11, 12, 66, 77, 63, 81, 82, 13]  # 1. Forest + 2. Grassland and shrubland (no rocky outcrop)

# Masks: 1 where the class belongs to the group, masked everywhere else
agri = lulc.remap(AGRI_CLASSES, [1] * len(AGRI_CLASSES)).rename("mask")
veg = lulc.remap(VEG_CLASSES, [1] * len(VEG_CLASSES)).rename("mask")

# Area of each group (km2)
for name, mask in [("Agriculture", agri), ("Natural vegetation", veg)]:
    area = mask.multiply(ee.Image.pixelArea()).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=basin, scale=100, maxPixels=1e13, tileScale=4)
    print(f"{name:20s}: {area.getInfo()['mask'] / 1e6:,.0f} km2")

Map = geemap.Map()
Map.centerObject(roi, 8)
Map.add_basemap("SATELLITE")
Map.addLayer(veg, {"palette": ["#1a9641"]}, f"Natural vegetation {LULC_YEAR}")
Map.addLayer(agri, {"palette": ["#e6b800"]}, f"Agriculture {LULC_YEAR}")
Map.addLayer(roi.style(color="red", fillColor="00000000", width=1), {}, "Watershed")
Map.add_legend(title=f"MapBiomas {LULC_YEAR}",
               legend_dict={"Agriculture": "#e6b800", "Natural vegetation": "#1a9641"})
Map

Now we apply each mask to every month of the **last 5 years** and compute the mean P, ET and
P - ET of the masked pixels. One function does it for any mask, so we call it twice.

> **Scale.** The masks are 30 m, MOD16 is 500 m and CHIRPS ~5.5 km. We reduce at **100 m** to
> keep the shape of the fields without too much computation. One MOD16 pixel mixes a field with
> its surroundings - for field-level ET, Landsat ET (section 4.3) is the right tool.

In [ ]:
# Last 5 years of the monthly collection
last5 = monthly.filter(ee.Filter.gte("year", END_YEAR - 4))


def annual_balance(mask):
    # Mean P, ET and P - ET of the masked pixels, for each month
    def masked_mean(image):
        stats = image.updateMask(mask).reduceRegion(reducer=ee.Reducer.mean(), geometry=basin,
                                                    scale=100, maxPixels=1e13, tileScale=4)
        return ee.Feature(None, stats).set("date", image.date().format("YYYY-MM-dd"))

    table = geemap.ee_to_df(ee.FeatureCollection(last5.map(masked_mean)))
    table["date"] = pd.to_datetime(table["date"])

    # Annual totals (mm/year)
    annual_table = table.groupby(table["date"].dt.year)[["precipitation", "ET", "water_balance"]].sum()
    annual_table.index.name = "year"
    return annual_table


annual_agri = annual_balance(agri)
annual_veg = annual_balance(veg)

print("Agriculture (mm/year)")
display(annual_agri.round(0))
print("Natural vegetation (mm/year)")
display(annual_veg.round(0))

In [ ]:
# P, ET and P - ET per year: agriculture (left) and natural vegetation (right)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)

for ax, table, title in [(axes[0], annual_agri, "Agriculture"),
                         (axes[1], annual_veg, "Natural vegetation")]:
    table.plot.bar(ax=ax, rot=0, width=0.8, color=["#25a6d7", "#757373", "#eb9800"], legend=False)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(f"{title} (MapBiomas {LULC_YEAR})")
    ax.set_xlabel("Year")
    ax.set_ylabel("mm/year")
    ax.spines[["top", "right"]].set_visible(False)

axes[0].legend(["Precipitation (CHIRPS)", "ET (MOD16)", "Water balance (P - ET)"], frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# Mean of the last 5 years (mm/year): agriculture, natural vegetation and the whole watershed
pd.DataFrame({
    "Agriculture": annual_agri.mean(),
    "Natural vegetation": annual_veg.mean(),
    "Whole watershed": annual.loc[END_YEAR - 4:].mean(),
}).round(0)

**How to read it.** Compare the columns. Does agriculture evaporate more or less than the
natural vegetation? Is the annual P - ET positive (surplus) or negative? A negative balance
means the surface uses more water than the rain gives in that year - the difference comes from
soil storage, groundwater or **irrigation**. Remember that both groups receive almost the same
rain (CHIRPS is ~5.5 km), so the differences in P - ET come mainly from **ET**.

> **Note.** The masks are fixed to one year (`LULC_YEAR`). Fields change from year to year; for a
> stricter analysis, use the MapBiomas band of each year as the mask of that year.

---

## 4.5 - Ejercicio / Exercise

1. **Another precipitation product.** Replace CHIRPS with **GSMaP**
   (`JAXA/GPM_L3/GSMaP/v8/operational`, band `hourlyPrecipRate`, mm/**hour**) or **ERA5-Land**
   (`ECMWF/ERA5_LAND/MONTHLY_AGGR`, band `total_precipitation_sum`, in **metres** - multiply by
   1000). Plot the annual totals of both products side by side. How different are they?
2. **Wet versus dry season.** Using `df`, compute the mean P - ET of December-March and of
   May-September. What fraction of the annual rain falls in the wet season?
3. **Water balance by land cover.** Use the classification map of session 1 (or ESA WorldCover)
   and compute the mean annual ET of each class. Which class evaporates the most?
4. **Longer record.** Change the period to 2001-2025. Does the trend of P keep the same sign?

**Entregable / Deliverable**: the notebook with the charts of questions 1 and 4 and a short
paragraph on what the P - ET balance says about the water available to the lake.

---

## Referencias / References

**Water balance, precipitation and ET**

- Funk, C. et al. (2015). The climate hazards infrared precipitation with stations - a new
  environmental record for monitoring extremes. *Scientific Data*, 2, 150066.
  <https://doi.org/10.1038/sdata.2015.66> (CHIRPS)
- Kubota, T. et al. (2020). Global Satellite Mapping of Precipitation (GSMaP) products in the
  GPM era. In *Satellite Precipitation Measurement*, Springer.
  <https://doi.org/10.1007/978-3-030-24568-9_20>
- Huffman, G. J. et al. (2020). Integrated Multi-satellite Retrievals for the Global
  Precipitation Measurement (GPM) mission (IMERG). In *Satellite Precipitation Measurement*,
  Springer. <https://doi.org/10.1007/978-3-030-24568-9_19>
- Muñoz-Sabater, J. et al. (2021). ERA5-Land: a state-of-the-art global reanalysis dataset for
  land applications. *Earth System Science Data*, 13, 4349-4383.
  <https://doi.org/10.5194/essd-13-4349-2021>
- Mu, Q., Zhao, M., Running, S. W. (2011). Improvements to a MODIS global terrestrial
  evapotranspiration algorithm. *Remote Sensing of Environment*, 115(8), 1781-1800.
  <https://doi.org/10.1016/j.rse.2011.02.019> (MOD16)
- Senay, G. B. et al. (2013). Operational evapotranspiration mapping using remote sensing and
  weather datasets: a new parameterization for the SSEB approach. *JAWRA*, 49(3), 577-591.
  <https://doi.org/10.1111/jawr.12057> (SSEBop)
- Abatzoglou, J. T. et al. (2018). TerraClimate, a high-resolution global dataset of monthly
  climate and climatic water balance from 1958-2015. *Scientific Data*, 5, 170191.
  <https://doi.org/10.1038/sdata.2017.191>

**Data and documentation**

- CHIRPS daily: <https://developers.google.com/earth-engine/datasets/catalog/UCSB-CHG_CHIRPS_DAILY>
- CHIRPS v3: <https://developers.google.com/earth-engine/datasets/catalog/UCSB-CHC_CHIRPS_V3_DAILY_RNL>
- GSMaP v8: <https://developers.google.com/earth-engine/datasets/catalog/JAXA_GPM_L3_GSMaP_v8_operational>
- IMERG V07: <https://developers.google.com/earth-engine/datasets/catalog/NASA_GPM_L3_IMERG_V07>
- ERA5-Land monthly: <https://developers.google.com/earth-engine/datasets/catalog/ECMWF_ERA5_LAND_MONTHLY_AGGR>
- MOD16A2GF: <https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MOD16A2GF>
- ET datasets in GEE: <https://developers.google.com/earth-engine/datasets/tags/evapotranspiration>
- Landsat Collection 2 Provisional Actual ET (USGS):
  <https://www.usgs.gov/landsat-missions/landsat-collection-2-provisional-actual-evapotranspiration-science-product>

**Book (open access)**

- Cardille, J.A., Crowley, M.A., Saah, D., Clinton, N.E. (eds.) (2023). *Cloud-Based Remote
  Sensing with Google Earth Engine: Fundamentals and Applications*. Springer.
  <https://www.eefabook.org/>